# FIGS-W — Training Diagnostics

Dataset balance, per-lead-band held-out skill for wildfire occurrence,
calibration curves, and the conditional fire-size distribution.
Mirrors FIGS notebook 2.

In [ ]:
# Run with the `met` conda env kernel.
import warnings; warnings.filterwarnings('ignore')
import sys, numpy as np, pandas as pd, matplotlib.pyplot as plt
sys.path.insert(0, '..')   # so `import figs_w` works from notebooks/
from figs_w import config as C

In [ ]:
from figs.data.dataset import read_dataset
from figs_w.data.dataset import META_COLS, LABEL_COLS
from figs_w.model.predict import _band_tags
from figs.model.wrapper import GBDTModel
from figs.model.calibrate import Calibrator, reliability_ci, low_dense_edges
import json, glob

DATA   = 'Data/figs_w/processed/fw.parquet'   # the built FIGS-W matrix
MODELS = C.MODELS
VAL_SAMPLE = 400_000
SIZE_LABELS = C.INTENSITY_BINS['wildfire']['labels']
EDGES = np.asarray(C.INTENSITY_BINS['wildfire']['edges'], float)
# size bins are DERIVED from the raw wildfire_size column at analysis time
def size_bin(sz): return (np.searchsorted(EDGES, np.asarray(sz, float), side='right') - 1).clip(0, len(SIZE_LABELS)-1)

## Split sizes and class balance (metadata only)

In [ ]:
meta = read_dataset(DATA, columns=META_COLS + LABEL_COLS)
print(meta.split.value_counts())
print(f"wildfire positives : {100*meta.wildfire.mean():.3f}% ({int(meta.wildfire.sum()):,})")
sz = meta['wildfire_size'].to_numpy(float); sz = sz[np.isfinite(sz) & (sz > 0)]
hist = np.bincount(size_bin(sz), minlength=len(SIZE_LABELS)) / max(len(sz), 1)
print(f'conditional size distribution (among {len(sz):,} wildfire cells with known size):')
for i, lab in enumerate(SIZE_LABELS): print(f'  {lab:>10s}: {100*hist[i]:5.1f}%')

## Positive rate by lead band

In [ ]:
if 'fxx' in meta:
    rows = []
    for b in C.LEAD_BANDS:
        sub = meta[(meta.fxx>=b.fmin)&(meta.fxx<=b.fmax)]
        if len(sub): rows.append(dict(band=b.name, n=len(sub), wildfire=sub.wildfire.mean()))
    bp = pd.DataFrame(rows).set_index('band'); display(bp)

## Held-out validation skill per band (occurrence)

Loads each band's calibrated model, scores the validation split, reports AUC / AUPRC.

In [ ]:
from sklearn.metrics import roc_auc_score, average_precision_score
feat_cols = json.loads((MODELS/'feature_cols.json').read_text())
RAW = ['wildfire']
def load_val(b):
    f = [('fxx','>=',b.fmin),('fxx','<=',b.fmax),('split','==','validation')]
    d = read_dataset(DATA, filters=f, columns=feat_cols + RAW)
    return d.sample(VAL_SAMPLE, random_state=0) if len(d) > VAL_SAMPLE else d
rows = []
for b in C.LEAD_BANDS:
    if not (MODELS/f'hazard_wildfire_{b.name}.pkl').exists(): continue
    d = load_val(b)
    if d.empty or d.wildfire.nunique() < 2: continue
    X = d[feat_cols].to_numpy(np.float32)
    y = {'wildfire': d.wildfire.to_numpy(int)}
    r = dict(band=b.name, n=len(d), pos=int(d.wildfire.sum()))
    for name in ('wildfire',):
        m = MODELS/f'hazard_{name}_{b.name}.pkl'
        if not m.exists() or len(np.unique(y[name])) < 2: continue
        p = GBDTModel.load(m).predict_pos(X)
        cp = MODELS/f'calib_{name}_{b.name}.pkl'
        if cp.exists(): p = Calibrator.load(cp).transform(p)
        r[f'{name}_auc']   = roc_auc_score(y[name], p)
        r[f'{name}_auprc'] = average_precision_score(y[name], p)
    rows.append(r)
skill = pd.DataFrame(rows).set_index('band'); display(skill)

## AUC / AUPRC vs lead band

In [ ]:
if len(skill):
    fig, ax = plt.subplots(1, 2, figsize=(13,4))
    for c in ('wildfire_auc',):
        if c in skill: ax[0].plot(skill.index, skill[c], 'o-', label=c)
    ax[0].set_title('AUC by band'); ax[0].set_ylim(0.5,1.0); ax[0].legend(); ax[0].grid(alpha=.3)
    for c in ('wildfire_auprc',):
        if c in skill: ax[1].plot(skill.index, skill[c], 'o-', label=c)
    ax[1].set_title('AUPRC by band'); ax[1].legend(); ax[1].grid(alpha=.3)
    for a in ax: a.tick_params(axis='x', rotation=45)
    plt.tight_layout(); plt.show()

## Reliability (calibration) — all-band ensemble, held-out TEST split, weighted

In [ ]:
tags = _band_tags(MODELS)
f = [('split','==','test')]
dt = read_dataset(DATA, filters=f, columns=feat_cols + ['wildfire','weight'])
dt = dt.sample(VAL_SAMPLE, random_state=0) if len(dt) > VAL_SAMPLE else dt
Xt = dt[feat_cols].to_numpy(np.float32)
ytest = {'wildfire': dt.wildfire.to_numpy()}
def ensemble_p(kind):
    ps = []
    for t in tags:
        m = MODELS/f'hazard_{kind}_{t}.pkl'
        if not m.exists(): continue
        p = GBDTModel.load(m).predict_pos(Xt)
        cp = MODELS/f'calib_{kind}_{t}.pkl'
        if cp.exists(): p = Calibrator.load(cp).transform(p)
        ps.append(p)
    return np.mean(ps, axis=0) if ps else None
edges = np.array([0,0.02,0.05,0.10,0.20,0.35,0.6,1.0])
fig, ax = plt.subplots(figsize=(6,6)); ax.plot([0,1],[0,1],'k--',lw=.8)
for kind in ('wildfire',):
    p = ensemble_p(kind)
    if p is None: continue
    mid, obs, lo, hi, _w, _n = reliability_ci(p, ytest[kind], sample_weight=dt.weight.to_numpy(), edges=edges)
    ax.errorbar(mid, obs, yerr=[obs-lo, hi-obs], marker='o', capsize=3, label=kind)
ax.set_xlabel('forecast probability'); ax.set_ylabel('observed frequency')
ax.set_title('FIGS-W reliability (test, weighted)'); ax.legend(); ax.grid(alpha=.3); plt.show()

## Conditional fire-size distribution: observed vs predicted (the CIG target)

In [ ]:
smp = MODELS/f'intensity_wildfire_{tags[0]}.pkl' if tags else None
if smp and smp.exists():
    d = read_dataset(DATA, filters=[('split','==','validation')], columns=feat_cols+['wildfire_size'])
    d = d[np.isfinite(d.wildfire_size) & (d.wildfire_size > 0)]
    d = d.sample(VAL_SAMPLE, random_state=0) if len(d) > VAL_SAMPLE else d
    nb_ = len(SIZE_LABELS)
    obs = np.bincount(size_bin(d.wildfire_size.to_numpy()), minlength=nb_)[:nb_]; obs = obs/obs.sum()
    # predicted conditional distribution averaged over the same cells (all bands)
    X = d[feat_cols].to_numpy(np.float32); acc = np.zeros(nb_); nseen = 0
    for t in tags:
        sp = MODELS/f'intensity_wildfire_{t}.pkl'
        if not sp.exists(): continue
        m = GBDTModel.load(sp); pr = m.predict_proba(X); full = np.zeros((len(X), nb_))
        for j, c in enumerate(np.asarray(m.classes_).astype(int)):
            if 0 <= c < nb_: full[:, c] = pr[:, j]
        acc += full.mean(axis=0); nseen += 1
    pred = acc/nseen if nseen else np.full(nb_, np.nan)
    x = np.arange(nb_); w=0.4
    fig, ax = plt.subplots(figsize=(8,4))
    ax.bar(x-w/2, obs, w, label='observed'); ax.bar(x+w/2, pred, w, label='predicted')
    ax.set_xticks(x); ax.set_xticklabels([f'{l} ac' for l in SIZE_LABELS], rotation=20)
    ax.set_ylabel('frequency'); ax.legend(); ax.set_title('conditional fire size: observed vs predicted'); plt.show()